# 01 — Coleta e qualidade dos dados

Pré-projeto *Preço do Café e da Soja × Dólar × Clima*. Este notebook executa (ou reaproveita) a coleta de cada fonte e inspeciona os dados brutos. Toda a lógica está em `src/`; aqui apenas chamamos as funções.

| Fonte | Técnica | Módulo |
|---|---|---|
| Notícias Agrícolas | Web scraping (requests + BeautifulSoup) | `src/coleta/noticias_agricolas.py` |
| CEPEA/ESALQ | Planilha (download manual) | `src/coleta/cepea.py` |
| Banco Central (SGS) | API REST | `src/coleta/bcb.py` |
| NASA POWER | API REST | `src/coleta/nasa_power.py` |
| INMET | Download de arquivos | `src/coleta/inmet.py` |
| Yahoo Finance | API (yfinance) | `src/coleta/mercado_internacional.py` |

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import config
from src.analise.estilo import aplicar_estilo_matplotlib

aplicar_estilo_matplotlib()
pd.set_option("display.max_columns", 50)

## Coleta
Defina `COLETAR = True` para baixar novamente. O backfill do histórico do Notícias Agrícolas é longo (~3 h) e deve ser rodado pelo terminal: `python -m src.coleta.noticias_agricolas --backfill`.

In [2]:
COLETAR = False
if COLETAR:
    from src.pipeline import coleta
    coleta(backfill=False, inmet=True)

## Dados brutos

In [3]:
cotacoes = pd.read_csv(config.RAW_NA / "cotacoes.csv", parse_dates=["data"])
print(cotacoes.groupby("produto")["data"].agg(["min", "max", "count"]))
cotacoes.tail()

               min        max  count
produto                             
cafe    2020-01-02 2026-11-12   1708
soja    2020-01-02 2026-11-12   1707


,data,produto,preco,variacao_pct,fonte,coletado_em
3410,2026-11-10,soja,159.87,0.79,noticias_agricolas,2026-11-13T10:06:00
3411,2026-11-11,cafe,1593.28,-1.12,noticias_agricolas,2026-11-13T10:06:03
3412,2026-11-11,soja,160.41,0.34,noticias_agricolas,2026-11-13T10:06:07
3413,2026-11-12,cafe,1619.43,1.64,noticias_agricolas,2026-11-13T10:06:10
3414,2026-11-12,soja,162.31,1.18,noticias_agricolas,2026-11-13T10:06:13


In [4]:
dolar = pd.read_csv(config.RAW_BCB / "dolar.csv", parse_dates=["data"])
ipca = pd.read_csv(config.RAW_BCB / "ipca.csv", parse_dates=["data"])
dolar.describe()

,data,dolar
count,1713,1713.000000
mean,2023-06-01 05:24:29.001751296,5.265971
min,2020-01-02 00:00:00,4.021300
25%,2021-09-16 00:00:00,5.061100
50%,2023-06-01 00:00:00,5.246500
75%,2025-02-12 00:00:00,5.477300
max,2026-10-28 00:00:00,6.208600
std,NaN,0.334588


In [5]:
nasa = pd.concat([pd.read_csv(p, parse_dates=["data"]) for p in sorted(config.RAW_NASA.glob("*.csv"))])
nasa.replace(-999, pd.NA).groupby("regiao").describe().T.head(15)

regiao          cerrado_mineiro       matas_de_minas         sul_de_minas
data count                 2494                 2494                 2494
     mean   2023-05-31 12:00:00  2023-05-31 12:00:00  2023-05-31 12:00:00
     min    2020-01-01 00:00:00  2020-01-01 00:00:00  2020-01-01 00:00:00
     25%    2021-09-15 06:00:00  2021-09-15 06:00:00  2021-09-15 06:00:00
     50%    2023-05-31 12:00:00  2023-05-31 12:00:00  2023-05-31 12:00:00
     75%    2025-02-12 18:00:00  2025-02-12 18:00:00  2025-02-12 18:00:00
     max    2026-10-29 00:00:00  2026-10-29 00:00:00  2026-10-29 00:00:00

In [6]:
estacoes = config.RAW_INMET / "estacoes_selecionadas.csv"
pd.read_csv(estacoes) if estacoes.exists() else "INMET ainda não coletado"

,regiao,ano,estacao,codigo,lat,lon,distancia_km,arquivo
0,sul_de_minas,2020,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2020.csv
1,cerrado_mineiro,2020,PATROCINIO,A523,-18.996684,-46.985935,5.9,cerrado_mineiro_A523_2020.csv
2,matas_de_minas,2020,MANHUACU,A556,-20.263333,-42.182778,15.6,matas_de_minas_A556_2020.csv
3,sul_de_minas,2021,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2021.csv
4,cerrado_mineiro,2021,PATROCINIO,A523,-18.996684,-46.985935,5.9,cerrado_mineiro_A523_2021.csv
5,matas_de_minas,2021,MANHUACU,A556,-20.263333,-42.182778,15.6,matas_de_minas_A556_2021.csv
6,sul_de_minas,2022,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2022.csv
7,cerrado_mineiro,2022,PATROCINIO,A523,-18.996684,-46.985935,5.9,cerrado_mineiro_A523_2022.csv
8,matas_de_minas,2022,MANHUACU,A556,-20.263333,-42.182778,15.6,matas_de_minas_A556_2022.csv
9,sul_de_minas,2023,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2023.csv


## Tratamento, integração e relatório de qualidade
Gera `data/processed/base_integrada.csv` e `reports/qualidade.md` (lacunas, outliers por IQR, validação NASA × INMET e Notícias Agrícolas × CEPEA).

In [7]:
from src.pipeline import tratamento
base = tratamento()
base.shape

10:07:01 INFO    src.tratamento.limpeza: interim/dolar.csv: 1713 linhas


10:07:01 INFO    src.tratamento.limpeza: interim/ipca.csv: 81 linhas


10:07:01 WARNING src.coleta.cepea: CEPEA cafe ausente. Baixe em https://www.cepea.org.br/br/indicador/series/cafe.aspx?id=23 e salve como data/raw/cepea/cafe.xls


10:07:01 WARNING src.coleta.cepea: CEPEA soja ausente. Baixe em https://www.cepea.org.br/br/indicador/series/soja.aspx?id=92 e salve como data/raw/cepea/soja.xls


10:07:01 INFO    src.tratamento.limpeza: interim/precos.csv: 3415 linhas


10:07:01 INFO    src.tratamento.limpeza: interim/internacional.csv: 1719 linhas


... (4 outputs left out)

10:07:02 WARNING src.coleta.cepea: CEPEA cafe ausente. Baixe em https://www.cepea.org.br/br/indicador/series/cafe.aspx?id=23 e salve como data/raw/cepea/cafe.xls


10:07:02 WARNING src.coleta.cepea: CEPEA soja ausente. Baixe em https://www.cepea.org.br/br/indicador/series/soja.aspx?id=92 e salve como data/raw/cepea/soja.xls


10:07:02 INFO    src.tratamento.qualidade: reports/qualidade.md gerado


(1723, 67)

In [8]:
from IPython.display import Markdown
Markdown((config.REPORTS / "qualidade.md").read_text(encoding="utf-8"))

# Relatório de qualidade dos dados

Gerado a partir de `data/processed/base_integrada.csv` (1723 dias úteis).

## Cobertura por variável

| variavel | inicio | fim | observacoes | faltantes_pct |
|---|---|---|---|---|
| preco_cafe | 2020-01-02 | 2026-11-12 | 1723 | 0.0 |
| preco_soja | 2020-01-02 | 2026-11-12 | 1723 | 0.0 |
| dolar | 2020-01-02 | 2026-11-03 | 1716 | 0.4 |
| cafe_ice_usd_saca | 2020-01-02 | 2026-11-04 | 1717 | 0.3 |
| soja_cbot_usd_saca | 2020-01-02 | 2026-11-04 | 1717 | 0.3 |
| ipca_mensal | 2020-01-02 | 2026-09-30 | 1694 | 1.7 |
| chuva_mm | 2020-01-02 | 2026-10-21 | 1708 | 0.9 |
| temp_med | 2020-01-02 | 2026-10-21 | 1708 | 0.9 |

## Maiores lacunas — preco_cafe

| de | ate | dias |
|---|---|---|
| 2020-02-21 | 2020-02-26 | 5 |
| 2021-02-12 | 2021-02-17 | 5 |
| 2022-02-25 | 2022-03-02 | 5 |
| 2023-02-17 | 2023-02-22 | 5 |
| 2024-02-09 | 2024-02-14 | 5 |

## Outliers de variação diária (IQR, k=3) — cafe

4 dias atípicos de 1722. Maiores:


| data | variacao_1d_pct |
|---|---|
| 2026-07-06 | 9.24 |
| 2021-07-22 | 7.06 |
| 2024-05-02 | -6.88 |
| 2021-07-21 | 6.41 |


Verificar se coincidem com eventos reais (ex.: geada de jul/2021) antes de remover.

## Maiores lacunas — preco_soja

| de | ate | dias |
|---|---|---|
| 2020-02-21 | 2020-02-26 | 5 |
| 2021-02-12 | 2021-02-17 | 5 |
| 2022-02-25 | 2022-03-02 | 5 |
| 2023-02-17 | 2023-02-22 | 5 |
| 2024-02-09 | 2024-02-14 | 5 |

## Outliers de variação diária (IQR, k=3) — soja

9 dias atípicos de 1722. Maiores:


| data | variacao_1d_pct |
|---|---|
| 2021-01-05 | 5.2 |
| 2020-11-12 | -5.09 |
| 2022-10-03 | -4.41 |
| 2020-12-09 | 4.31 |
| 2021-06-30 | 4.2 |
| 2022-04-01 | -4.14 |
| 2020-09-21 | 3.9 |
| 2020-11-09 | -3.82 |
| 2020-11-10 | 3.77 |


Verificar se coincidem com eventos reais (ex.: geada de jul/2021) antes de remover.

## Validação: Notícias Agrícolas × CEPEA

_sem dados_

## Validação: NASA POWER × INMET

| regiao | dias_em_comum | corr_chuva_mensal | corr_temp_med_diaria | corr_temp_min_diaria | vies_temp_min_C |
|---|---|---|---|---|---|
| cerrado_mineiro | 2465 | 0.947 | 0.845 | 0.843 | 1.85 |
| matas_de_minas | 2465 | 0.896 | 0.893 | 0.862 | 0.56 |
| sul_de_minas | 2465 | 0.855 | 0.931 | 0.928 | 0.79 |


In [9]:
base.isna().mean().sort_values(ascending=False).head(15).round(3)

chuva_180d                     0.071
chuva_90d                      0.040
chuva_90d_sul_de_minas         0.040
chuva_90d_cerrado_mineiro      0.040
chuva_90d_matas_de_minas       0.040
anomalia_chuva_90d             0.040
cafe_var_fut_63d               0.037
soja_var_fut_63d               0.037
soja_mm60                      0.026
cafe_mm60                      0.026
chuva_30d_matas_de_minas       0.019
temp_max_30d_matas_de_minas    0.019
temp_min_30d_matas_de_minas    0.019
chuva_30d_sul_de_minas         0.019
temp_max_30d_sul_de_minas      0.019
dtype: float64